# Whole Loan Total Return Swaps in LUSID

| Section | Topic |
|---|---|
| 1 | Instrument creation |
| 2 | Recipe |
| 3 | Portfolio and transactions |
| 4 | Valuation |
| 5 | Accrued interest |
| 6 | Maturity as an event |

## The instrument

A whole loan TRS has two legs pulling in opposite directions: one side carries the return on a
**directly-originated loan** (the asset leg), and the other pays a flat financing cost (the
funding leg):

    asset leg     the referenced loan, via ReferenceInstrument
    funding leg   a FixedLeg paying a flat financing rate

The loan itself is modelled as a `FlexibleLoan`. Its schedule can be fixed or floating -- here
it's fixed, with the coupon stepping up at scheduled reset dates over the loan's life.

## A constraint to know before you hit it

**Set the portfolio's holding recipe when you create it -- there's no way to add it later.** Any
portfolio that holds, or even just references, a `FlexibleLoan` needs a recipe attached at
creation time, so LUSID's valuation engine has something to resolve against. You can't patch one
onto an existing portfolio afterwards -- if a portfolio was created without one, it has to be
deleted and recreated.

---
## Setup

In [ ]:
import os
import json
import certifi
os.environ.setdefault("SSL_CERT_FILE", certifi.where())

from datetime import datetime, timezone, timedelta
import pandas as pd

import lusid
import lusid.models as m
from lusid.extensions import (
    SyncApiClientFactory, SecretsFileConfigurationLoader, EnvironmentVariablesConfigurationLoader)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.options.display.float_format = "{:,.2f}".format

SECRETS_PATH = os.getenv("FBN_SECRETS_PATH") or (
    "secrets.json" if os.path.exists("secrets.json") else None)
config_loaders = ([SecretsFileConfigurationLoader(SECRETS_PATH)] if SECRETS_PATH
                   else [EnvironmentVariablesConfigurationLoader()])

factory = SyncApiClientFactory(config_loaders=config_loaders)


def api(cls):
    return factory.build(cls)


instruments_api   = api(lusid.InstrumentsApi)
txn_portfolio_api = api(lusid.TransactionPortfoliosApi)
portfolios_api    = api(lusid.PortfoliosApi)
quotes_api        = api(lusid.QuotesApi)
recipes_api       = api(lusid.ConfigurationRecipeApi)
aggregation_api   = api(lusid.AggregationApi)

meta = api(lusid.ApplicationMetadataApi).get_lusid_versions()
href = meta.links[0].href
print("Domain      :", href[:href.find("/app/")] if "/app/" in href else href)
print("API version :", meta.build_version)

Domain      : https://fbn-tejan.lusid.com
API version : 0.6.16597.0


---
## Configuration

The loan's rate history is five contiguous periods, each at its own coupon. The valuation date
sits inside the fourth period, after three of the four step-ups have already kicked in. The
swap itself only has a single reset, at maturity -- the loan's own periods are what carry all
the rate detail.

In [2]:
def d(year, month, day):
    return datetime(year, month, day, tzinfo=timezone.utc)


def upsert(key, name, client_internal, definition):
    """Upsert one instrument and return its LUID."""
    resp = instruments_api.upsert_instruments(scope=SCOPE, request_body={
        key: m.InstrumentDefinition(
            name=name,
            identifiers={"ClientInternal": m.InstrumentIdValue(value=client_internal)},
            definition=definition)})
    assert not resp.failed, list(resp.failed.values())[0].detail
    return resp.values[key].lusid_instrument_id


def mastered(luid):
    """Reference an instrument that already exists in the master."""
    return m.MasteredInstrument(
        instrument_type="MasteredInstrument",
        identifiers={"Instrument/default/LusidInstrumentId": luid})


def recreate_portfolio(code, display_name, base_currency, created, recipe=None):
    """Create the portfolio, replacing any earlier run so the book starts empty."""
    request = m.CreateTransactionPortfolioRequest(
        display_name=display_name, code=code, base_currency=base_currency,
        created=created, instrument_scopes=[SCOPE],
        instrument_event_configuration=None if recipe is None else
        m.InstrumentEventConfiguration(
            transaction_template_scopes=["default"],
            recipe_id=m.ResourceId(scope=SCOPE, code=recipe)))
    try:
        txn_portfolio_api.create_portfolio(
            scope=SCOPE, create_transaction_portfolio_request=request)
        print(f"Created {SCOPE}/{code}")
    except lusid.ApiException as e:
        if "PortfolioWithIdAlreadyExists" not in str(getattr(e, "body", "")):
            raise
        portfolios_api.delete_portfolio(scope=SCOPE, code=code)
        txn_portfolio_api.create_portfolio(
            scope=SCOPE, create_transaction_portfolio_request=request)
        print(f"Recreated {SCOPE}/{code}")


def upsert_price(luid, price, effective, currency):
    """One Price/mid quote, keyed on the instrument's LUID."""
    quotes_api.upsert_quotes(scope=SCOPE, request_body={
        f"{luid}-{effective:%Y%m%d}": m.UpsertQuoteRequest(
            quote_id=m.QuoteId(
                quote_series_id=m.QuoteSeriesId(
                    provider="Lusid", instrument_id=luid,
                    instrument_id_type="LusidInstrumentId",
                    quote_type="Price", field="mid"),
                effective_at=effective.isoformat()),
            metric_value=m.MetricValue(value=price, unit=currency))})


def value(portfolio, effective, metrics, currency, group_by=None):
    """Run the recipe over one portfolio and return the result as a DataFrame."""
    request = m.ValuationRequest(
        recipe_id=m.ResourceId(scope=SCOPE, code=RECIPE),
        metrics=[m.AggregateSpec(key=k, op=op) for k, op in metrics],
        group_by=group_by or ["Instrument/default/Name"],
        report_currency=currency,
        portfolio_entity_ids=[m.PortfolioEntityId(
            scope=SCOPE, code=portfolio, portfolio_entity_type="SinglePortfolio")],
        valuation_schedule=m.ValuationSchedule(effective_at=effective.isoformat()))
    return pd.DataFrame(aggregation_api.get_valuation(valuation_request=request).data)


def transactions(portfolio, from_date, as_at):
    """The portfolio's own booked transactions over a date range, as a DataFrame."""
    txns = txn_portfolio_api.get_transactions(
        scope=SCOPE, code=portfolio,
        from_transaction_date=from_date.isoformat(),
        to_transaction_date=as_at.isoformat()).values
    if not txns:
        return pd.DataFrame(columns=["date", "type", "luid", "units", "consideration"])
    return pd.DataFrame([{
        "date": pd.Timestamp(t.transaction_date).strftime("%Y-%m-%d"),
        "type": t.type,
        "luid": t.instrument_uid,
        "units": t.units,
        "consideration": t.total_consideration.amount,
    } for t in txns]).sort_values(["date", "type"]).reset_index(drop=True)


SCOPE     = "WholeLoanTrsDemo"
RECIPE    = "whole-loan-trs-demo-recipe"
PORTFOLIO = "whole-loan-trs-demo-book"

LOAN_ID   = "DEMO-WESTBROOK-WL-01"
LOAN_DESC = "Demo Westbrook Multifamily Whole Loan"
SWAP_ID   = "DEMO-WESTBROOK-TRS-01"
SWAP_DESC = "Demo Westbrook Multifamily Whole Loan TRS"
CURRENCY  = "USD"

START     = d(2025, 1, 15)
MATURITY  = d(2030, 1, 15)
ASOF      = d(2026, 8, 1)

# (reset date, annual coupon) -- contiguous periods; the last runs to MATURITY.
RATE_STEPS = [
    (d(2025, 1, 15), 0.0725),
    (d(2025, 7, 15), 0.0750),
    (d(2026, 1, 15), 0.0775),
    (d(2026, 7, 15), 0.0800),
    (d(2027, 1, 15), 0.0825)]

FREQUENCY  = "6M"
DAY_COUNT  = "Actual360"

FUNDING_RATE = 0.0450       # flat -- FixedLeg, no fixing quotes needed

STRIKE      = 98.00         # points -- the swap's initial mark
LOAN_MARK   = 99.00         # points -- the loan's current mark
DENOM       = 100

INITIAL_PRICE = STRIKE / DENOM
NOTIONAL      = 1.0         # every FixedSchedule row on the loan shares this notional

QUANTITY = 10_000_000.00

CURRENT_RATE = next(rate for reset, rate in reversed(RATE_STEPS) if reset <= ASOF)

print(f"{SWAP_DESC}")
print(f"  asset   Receive  {LOAN_ID}")
for reset, rate in RATE_STEPS:
    flag = "  <- in force" if rate == CURRENT_RATE else ""
    print(f"      {reset:%Y-%m-%d}  {rate:.3%}{flag}")
print(f"  funding Pay      fixed {FUNDING_RATE:.3%}, {FREQUENCY} {DAY_COUNT}")
print(f"  {QUANTITY:,.0f} units, strike {STRIKE} -> mark {LOAN_MARK} on {ASOF:%Y-%m-%d}")
print(f"  asset leg return = {QUANTITY:,.0f} x ({LOAN_MARK} - {STRIKE}) / {DENOM} = "
      f"{QUANTITY * (LOAN_MARK - STRIKE) / DENOM:,.2f} {CURRENCY}")

Demo Westbrook Multifamily Whole Loan TRS
  asset   Receive  DEMO-WESTBROOK-WL-01
      2025-01-15  7.250%
      2025-07-15  7.500%
      2026-01-15  7.750%
      2026-07-15  8.000%  <- in force
      2027-01-15  8.250%
  funding Pay      fixed 4.500%, 6M Actual360
  10,000,000 units, strike 98.0 -> mark 99.0 on 2026-08-01
  asset leg return = 10,000,000 x (99.0 - 98.0) / 100 = 100,000.00 USD


---
# 1. Instrument creation

## 1a. The loan

The loan gets created on its own, separately from the swap, because the asset leg references it
by identifier rather than embedding it by value.

In [3]:
bounds = []
for i in range(len(RATE_STEPS) - 1):
    bounds.append((RATE_STEPS[i][0], RATE_STEPS[i + 1][0], RATE_STEPS[i][1]))
bounds.append((RATE_STEPS[-1][0], MATURITY, RATE_STEPS[-1][1]))

schedules = [
    m.FixedSchedule(
        schedule_type="FixedSchedule",
        start_date=start,
        maturity_date=end,
        flow_conventions=m.FlowConventions(
            currency=CURRENCY,
            payment_frequency=FREQUENCY,
            day_count_convention=DAY_COUNT,
            roll_convention=str(start.day),
            payment_calendars=[], reset_calendars=[]),
        coupon_rate=rate,
        notional=NOTIONAL,
        payment_currency=CURRENCY,
        stub_type="ShortBack")
    for start, end, rate in bounds]

loan = m.FlexibleLoan(
    instrument_type="FlexibleLoan",
    start_date=START,
    maturity_date=MATURITY,
    dom_ccy=CURRENCY,
    schedules=schedules)

LOAN_LUID = upsert("loan", LOAN_DESC, LOAN_ID, loan)
print(f"Loan : {LOAN_LUID}")

Loan : LUID_00003DG6


## 1b. The swap

The asset leg holds a `ReferenceInstrument` pointing at the loan's `ClientInternal` identifier. There's just one reset, at maturity, because the
loan's own periods already carry the rate detail; the swap doesn't need a second reset cadence
layered on top.

In [4]:
trs = m.TotalReturnSwap(
    instrument_type="TotalReturnSwap",
    start_date=START,
    maturity_date=MATURITY,
    asset_leg=m.AssetLeg(
        asset=m.ReferenceInstrument(
            instrument_type="ReferenceInstrument",
            instrument_id=LOAN_ID,
            instrument_id_type="ClientInternal",
            scope=SCOPE),
        pay_receive="Receive",
        initial_price=INITIAL_PRICE,
        reset_schedule=m.ResetSchedule(frequency=FREQUENCY, first_reset_date=MATURITY),
        income_policy="Reinvest"),
    funding_leg=m.FixedLeg(
        instrument_type="FixedLeg",
        start_date=START,
        maturity_date=MATURITY,
        notional=INITIAL_PRICE,
        leg_definition=m.LegDefinition(
            rate_or_spread=FUNDING_RATE,
            pay_receive="Pay",
            conventions=m.FlowConventions(
                currency=CURRENCY,
                payment_frequency=FREQUENCY,
                day_count_convention=DAY_COUNT,
                roll_convention=str(START.day),
                payment_calendars=[], reset_calendars=[]),
            stub_type="ShortBack",
            notional_exchange_type="None")))

TRS_LUID = upsert("trs", SWAP_DESC, SWAP_ID, trs)
print(f"Whole loan TRS : {TRS_LUID}")

Whole loan TRS : LUID_00003DGM


---
# 2. Recipe

This is where `FlexibleLoanPricer` comes in, registered here against `TotalReturnSwap` since
that's what's actually held in the portfolio. The `ReferenceInstrument` on the asset leg resolves to the loan's `LusidInstrumentId` before
pricing looks up a quote, so the loan's quote needs to be keyed by that LUID, not by its
`ClientInternal` identifier.

In [5]:
recipes_api.upsert_configuration_recipe(
    upsert_recipe_request=m.UpsertRecipeRequest(
        configuration_recipe=m.ConfigurationRecipe(
            scope=SCOPE, code=RECIPE,
            description="Whole loan TRS, FlexibleLoanPricer",
            market=m.MarketContext(
                market_rules=[
                    m.MarketDataKeyRule(
                        key="Quote.LusidInstrumentId.*", supplier="Lusid", data_scope=SCOPE,
                        quote_type="Price", field="mid", quote_interval="5D"),
                    m.MarketDataKeyRule(
                        key="Quote.ClientInternal.*", supplier="Lusid", data_scope=SCOPE,
                        quote_type="Price", field="mid", quote_interval="5D")],
                options=m.MarketOptions(
                    default_supplier="Lusid",
                    default_scope=SCOPE,
                    default_instrument_code_type="ClientInternal")),
            pricing=m.PricingContext(
                model_rules=[m.VendorModelRule(
                    supplier="Lusid", model_name="FlexibleLoanPricer",
                    instrument_type="TotalReturnSwap")],
                options=m.PricingOptions(
                    model_selection=m.ModelSelection(
                        library="Lusid", model="FlexibleLoanPricer"),
                    use_instrument_type_to_determine_pricer=True,
                    allow_partially_successful_evaluation=True)))))

print(f"Recipe: {SCOPE}/{RECIPE}")

Recipe: WholeLoanTrsDemo/whole-loan-trs-demo-recipe


---
# 3. Portfolio and transactions

This is where `instrument_event_configuration.recipe_id` has to be set -- right now, at creation,
for the reason covered above. `recreate_portfolio()`'s `recipe=` argument takes care of that.

The swap gets entered into rather than bought, so `totalConsideration` is zero.

In [6]:
recreate_portfolio(PORTFOLIO, "Whole Loan TRS Demo Book", CURRENCY, d(2025, 1, 1), recipe=RECIPE)

txn_portfolio_api.upsert_transactions(
    scope=SCOPE, code=PORTFOLIO,
    transaction_request=[m.TransactionRequest(
        transaction_id="BUY-TRS",
        type="Buy",
        instrument_identifiers={"Instrument/default/LusidInstrumentId": TRS_LUID},
        transaction_date=START.isoformat(),
        settlement_date=START.isoformat(),
        units=QUANTITY,
        transaction_price=m.TransactionPrice(price=0.0, type="Price"),
        total_consideration=m.CurrencyAndAmount(amount=0.0, currency=CURRENCY),
        source="default")])

display(transactions(PORTFOLIO, START, START))

Recreated WholeLoanTrsDemo/whole-loan-trs-demo-book


,date,type,luid,units,consideration
0,2025-01-15,Buy,LUID_00003DGM,"10,000,000.00",0.00


---
# 4. Valuation

Just one quote is needed here: the loan's own mark, against its LUID. `Valuation/Leg1/PV` gives
the asset leg's price return on its own, which is the figure to check against the loan's own mark
move.

In [7]:
upsert_price(LOAN_LUID, LOAN_MARK / DENOM, ASOF, CURRENCY)

METRICS = [("Instrument/default/Name", "Value"),
           ("Holding/default/Units",   "Sum"),
           ("Valuation/Leg1/PV",       "Sum"),
           ("Valuation/PV",            "Sum")]

result = value(PORTFOLIO, ASOF, METRICS, CURRENCY)
display(result)

leg1_pv = result.loc[result["Instrument/default/Name"] == SWAP_DESC,
                     "Sum(Valuation/Leg1/PV)"].iloc[0]
print(f"LUSID Leg1/PV {leg1_pv:,.2f}  vs  asset leg return "
      f"{QUANTITY * (LOAN_MARK - STRIKE) / DENOM:,.2f}")

,Instrument/default/Name,Sum(Holding/default/Units),Sum(Valuation/Leg1/PV),Sum(Valuation/PV)
0,Demo Westbrook Multifamily Whole Loan TRS,"10,000,000.00","100,000.00","-1,451,047.22"


LUSID Leg1/PV 100,000.00  vs  asset leg return 100,000.00


---
# 5. Accrued interest

`Valuation/Accrued` gives one combined number for the swap: the loan's accrual over its current
rate period, netted against the funding leg's accrual over its own coupon period.

In [8]:
ACCRUED = [("Instrument/default/Name", "Value"),
           ("Valuation/Accrued",       "Sum")]

display(value(PORTFOLIO, ASOF, ACCRUED, CURRENCY))

,Instrument/default/Name,Sum(Valuation/Accrued)
0,Demo Westbrook Multifamily Whole Loan TRS,"16,952.78"


---
# 6. Maturity as an event

Every `TotalReturnSwap` carries a `MaturityEvent` at its own `maturity_date`, with no extra setup
needed on the instrument for that. Query a window that spans past `MATURITY` and it shows up,
complete with a real, populated transaction -- the swap just closes out.

In [9]:
events_api = api(lusid.InstrumentEventsApi)

window_end = MATURITY + timedelta(days=5)

applicable = events_api.query_applicable_instrument_events(
    query_applicable_instrument_events_request=m.QueryApplicableInstrumentEventsRequest(
        window_start=ASOF.isoformat(),
        window_end=window_end.isoformat(),
        effective_at=window_end.isoformat(),
        portfolio_entity_ids=[m.PortfolioEntityId(
            scope=SCOPE, code=PORTFOLIO, portfolio_entity_type="SinglePortfolio")],
        forecasting_recipe_id=m.ResourceId(scope=SCOPE, code=RECIPE))).values

display(pd.DataFrame([{
    "event type": ev.instrument_event_type,
    "eligible balance": ev.eligible_balance,
    "status": ev.instrument_event_status,
} for ev in applicable]))

,event type,eligible balance,status
0,MaturityEvent,"10,000,000.00",Active


---
# Summary

1. A whole loan TRS points at the loan through a `ReferenceInstrument`, rather than holding it
   inline -- the loan is created separately and referenced by identifier.
2. A `FlexibleLoan`'s schedule can be fixed or floating. This one's fixed, with a coupon that
   steps up over the loan's life instead of staying flat.
3. It's priced under `FlexibleLoanPricer`, registered here against `TotalReturnSwap` since that's
   what's actually held -- a `FlexibleLoan` held directly would register the same model against
   `FlexibleLoan` instead.
4. The portfolio's holding recipe can only be set at creation, and there's no way around that --
   any `FlexibleLoan` in the dependency graph needs one attached, or valuation has nothing to
   resolve against. That same recipe wiring is also what makes the swap's own `MaturityEvent`
   forecastable, as shown in section 6.

In [10]:
print(f"Scope      : {SCOPE}")
print(f"Portfolio  : {SCOPE}/{PORTFOLIO}")
print(f"Recipe     : {SCOPE}/{RECIPE}")
print(f"Loan       : {LOAN_LUID}")
print(f"Swap       : {TRS_LUID}")

Scope      : WholeLoanTrsDemo
Portfolio  : WholeLoanTrsDemo/whole-loan-trs-demo-book
Recipe     : WholeLoanTrsDemo/whole-loan-trs-demo-recipe
Loan       : LUID_00003DG6
Swap       : LUID_00003DGM
